In [3]:
import os
import math
import random
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple
from datetime import datetime

import numpy as np
from pymongo import MongoClient, UpdateOne
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

@dataclass
class CFG:
    # Mongo
    host: str = "10.255.68.40"
    port: int = 27017
    username: str = ""  # empty means no auth
    password: str = ""
    db_name: str = "ejoow"
    src_collection: str = "2. US_sequence_catlen_gt1_with_catid"  # note trailing space
    out_collection: str = "user_tl_day_embeddings_v1"

    # Tokens / vocab
    pad_id: int = 0
    n_items: int = 431         # category_id in [1..431]
    mask_id: int = 432         # special token
    vocab_size: int = 433      # 0..432

    # Sequence / MLM
    max_len: int = 16
    mask_ratio: float = 0.5

    # Model
    d_model: int = 64
    n_heads: int = 2
    n_layers: int = 2
    dropout: float = 0.1

    # Train
    seed: int = 42
    batch_size: int = 512
    lr: float = 1e-3
    weight_decay: float = 1e-4
    epochs: int = 5
    num_workers: int = 4

    # Device
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

cfg = CFG()
cfg

client = MongoClient(cfg.host, cfg.port)
db = client[cfg.db_name]

src_col = db["3. user_cluster_frequency"]
out_col = db["4. user_cluster_vector"]

print("src count:", src_col.estimated_document_count())


src count: 67207


2️⃣ freq → dense vector 함수

In [5]:
def build_cluster_vector(freq_dict, k=50, normalize="prob"):
    """
    freq_dict: {"44": 2, "2": 1, ...}
    k: number of clusters
    normalize: "prob" | "logprob" | None
    """
    v = np.zeros(k, dtype=np.float32)

    for cid_str, cnt in freq_dict.items():
        cid = int(cid_str)
        if 0 <= cid < k:
            v[cid] = float(cnt)

    if normalize == "prob":
        s = v.sum()
        if s > 0:
            v /= s

    elif normalize == "logprob":
        v = np.log1p(v)
        s = v.sum()
        if s > 0:
            v /= s

    return v


3️⃣ 전체 컬렉션 벡터화 & 저장

In [6]:
out_col.create_index("user_id", unique=True)

ops = []
written = 0
NORMALIZE_MODE = "prob"   # or "logprob"

cursor = src_col.find({}, {
    "_id": 0,
    "user_id": 1,
    "freq": 1,
    "k": 1
})

for doc in tqdm(cursor, desc="Vectorizing user clusters"):
    user_id = doc["user_id"]
    freq = doc.get("freq", {})
    k = int(doc.get("k", 50))

    vec = build_cluster_vector(freq, k=k, normalize=NORMALIZE_MODE)

    out_doc = {
        "_id": user_id,
        "user_id": user_id,
        "k": k,
        "vector": vec.tolist(),
        "normalize": NORMALIZE_MODE,
        "nonzero": int((vec > 0).sum()),
        "updated_at": datetime.utcnow()
    }

    ops.append(UpdateOne(
        {"_id": user_id},
        {"$set": out_doc},
        upsert=True
    ))

    if len(ops) >= 5000:
        out_col.bulk_write(ops, ordered=False)
        written += len(ops)
        ops = []

if ops:
    out_col.bulk_write(ops, ordered=False)
    written += len(ops)

print("done. written:", written)
print("out count:", out_col.estimated_document_count())


Vectorizing user clusters: 0it [00:00, ?it/s]

done. written: 67207
out count: 67207


4️⃣ 저장 결과 빠른 검증

In [7]:
sample = out_col.find_one({})
print("user_id:", sample["user_id"])
print("k:", sample["k"])
print("nonzero:", sample["nonzero"])
print("vector (non-zero):")
for i, v in enumerate(sample["vector"]):
    if v > 0:
        print(i, v)


user_id: 1696853
k: 50
nonzero: 1
vector (non-zero):
24 1.0


5️⃣ k별 silhouette score 비교

In [ ]:
from sklearn.metrics import silhouette_score

SIL_SAMPLE = 10000
rng = np.random.RandomState(42)

silhouette_results = []
available_k = sorted(out_col.distinct("k"))

for k in available_k:
    docs = list(out_col.find(
        {"k": int(k)},
        {"_id": 0, "user_id": 1, "vector": 1}
    ))

    if len(docs) < 2:
        print(f"skip k={k}: sample 수가 부족합니다. n={len(docs)}")
        continue

    X = np.array([doc["vector"] for doc in docs], dtype=np.float32)
    labels = X.argmax(axis=1)

    unique_labels, counts = np.unique(labels, return_counts=True)
    if len(unique_labels) < 2:
        print(f"skip k={k}: 실제 할당된 클러스터가 1개뿐입니다.")
        continue

    sample_size = min(SIL_SAMPLE, len(X))
    if sample_size < len(X):
        sample_idx = rng.choice(len(X), size=sample_size, replace=False)
        X_eval = X[sample_idx]
        labels_eval = labels[sample_idx]
    else:
        X_eval = X
        labels_eval = labels

    if np.unique(labels_eval).shape[0] < 2:
        print(f"skip k={k}: 샘플링 후 클러스터가 1개만 남았습니다.")
        continue

    sil = silhouette_score(X_eval, labels_eval, metric="euclidean")

    row = {
        "k": int(k),
        "silhouette": float(sil),
        "n_users": int(len(X)),
        "n_clusters_used": int(len(unique_labels)),
        "min_cluster_size": int(counts.min()),
        "max_cluster_size": int(counts.max()),
    }
    silhouette_results.append(row)
    print(
        f"k={k:>3} | silhouette={sil:.4f} | "
        f"n_users={len(X)} | used_clusters={len(unique_labels)}"
    )

if not silhouette_results:
    raise RuntimeError("계산 가능한 silhouette 결과가 없습니다.")

silhouette_results = sorted(
    silhouette_results,
    key=lambda x: x["silhouette"],
    reverse=True,
)

best_result = silhouette_results[0]
print("\nBest silhouette by k")
print(best_result)

silhouette_results
